# small-signal-equation-extractor Tutorial

This notebook demonstrates the public `SSAS.solve(...)` interface. Each call solves one request, saves one Markdown file, and returns `None`. The input mode is always explicit: use `source="inline"` for netlist text and `source="file"` for a netlist path. Relative paths are resolved by `SSAS` from the notebook working directory.


In [1]:
from SSAS import SSAS

ssas = SSAS(output_dir="tutorial_outputs")


## 1. RC low-pass response

A request is written directly as `V(out) / V(in)`. The AC value only marks the source as nonzero; it is not substituted as a numeric amplitude.


In [2]:
rc_lowpass_netlist = """
V1 in 0 ac=1
R1 in out 1
C1 out 0 1
"""

ssas.solve(
    rc_lowpass_netlist,
    "V(out) / V(in)",
    "rc_lowpass.md",
    source="inline",
    show=True,
)

\[
H(s)=\frac{1}{s R_{1} C_{1}+1}
\]

## 2. Compact-device equivalent-circuit mode

`eq_circuit=True` selects equivalent-circuit mode for supported compact devices. For MOSFETs, the capacitance symbols are terminal-pair capacitors such as `C_{mn1,gs}` and `C_{mn1,gd}`. For bipolar junction transistors, the same option selects the equivalent-circuit stamp for that device type.


In [3]:
cs_netlist = """
VIn in 0 ac=1
RD out 0 1
Mn1 out in 0 0 nmos
"""

ssas.solve(
    cs_netlist,
    "V(out) / V(in)",
    "cs_equivalent_gain.md",
    source="inline",
    eq_circuit=True,
    body="on",
    show=True,
)

\[
H(s)=\frac{s C_{mn1,gd}-g_{mn1}}{s\left(C_{mn1,gd}+C_{mn1,ds}+C_{mn1,db}\right)+\frac{1}{R_{D}}+\frac{1}{r_{mn1}}}
\]

## 3. Compact-device coefficient mode

`eq_circuit=False` selects coefficient mode for supported compact devices. For MOSFETs, this is a BSIM4-compatible external-port coefficient stamp, where BSIM4 denotes Berkeley Short-channel Insulated-Gate Field-Effect Transistor Model version 4. The extractor stamps symbolic terminal-current and terminal-charge derivative coefficients; a wrapper may supply those coefficients after evaluating the device model. The `C` symbols in this MOSFET mode are charge-Jacobian entries, not two-terminal capacitors. For bipolar junction transistors, the same option selects the bipolar junction transistor differential coefficient stamp.


In [4]:
ssas.solve(
    cs_netlist,
    "V(out) / V(in)",
    "cs_coefficient_gain.md",
    source="inline",
    eq_circuit=False,
    body="on",
    show=True,
)

\[
H(s)=\frac{-s C_{mn1,g,d}-g_{mn1,gs,d}}{s C_{mn1,d,d}+\frac{1}{R_{D}}+g_{mn1,ds,d}}
\]

## 4. Zero-pole post-analysis

`zero_pole=True` does not replace the ordinary response. It first saves the same `H(s)` output and then appends a zero-pole analysis section. If analytic roots cannot be extracted, the Markdown records that status instead of raising an error.


In [5]:
ssas.solve(
    cs_netlist,
    "V(out) / V(in)",
    "cs_equivalent_zero_pole.md",
    source="inline",
    eq_circuit=True,
    body="on",
    zero_pole=True,
    show=True,
)

\[
H(s)=\frac{s C_{mn1,gd}-g_{mn1}}{s\left(C_{mn1,gd}+C_{mn1,ds}+C_{mn1,db}\right)+\frac{1}{R_{D}}+\frac{1}{r_{mn1}}}
\]

\[
\text{zero-pole analysis}
\]

\[
Z_{1}=s-\left(\frac{g_{mn1}}{C_{mn1,gd}}\right)
\]

\[
P_{1}=s-\left(\frac{-\left(\frac{1}{R_{D}}+\frac{1}{r_{mn1}}\right)}{C_{mn1,gd}+C_{mn1,ds}+C_{mn1,db}}\right)
\]

## 5. Single-probe and current-ratio requests

Use the same request-string syntax for single probes and ratios involving passive and source currents. MOSFET and bipolar junction transistor instance currents are not public current probes.


In [6]:
ssas.solve(
    cs_netlist,
    "I(RD)",
    "rd_current.md",
    source="inline",
    eq_circuit=True,
    body="on",
    show=True,
)

ssas.solve(
    cs_netlist,
    "I(RD) / V(in)",
    "current_gain.md",
    source="inline",
    eq_circuit=True,
    body="on",
    show=True,
)


\[
H(s)=\frac{v_{in} \left(s C_{mn1,gd}-g_{mn1}\right)}{s R_{D} \left(C_{mn1,gd}+C_{mn1,ds}+C_{mn1,db}\right)+1+\frac{R_{D}}{r_{mn1}}}
\]

\[
H(s)=\frac{s C_{mn1,gd}-g_{mn1}}{s R_{D} \left(C_{mn1,gd}+C_{mn1,ds}+C_{mn1,db}\right)+1+\frac{R_{D}}{r_{mn1}}}
\]

## 6. Removing selected terms

`zero_cap` removes a selected capacitance term. In equivalent-circuit mode this opens the selected equivalent capacitor. In coefficient mode it removes the selected device differential capacitance coefficient.


In [7]:
ssas.solve(
    cs_netlist,
    "V(out) / V(in)",
    "cs_without_cgd.md",
    source="inline",
    eq_circuit=True,
    body="on",
    zero_cap="C_mn1_gd",
    show=True,
)

\[
H(s)=-\frac{g_{mn1}}{s\left(C_{mn1,ds}+C_{mn1,db}\right)+\frac{1}{R_{D}}+\frac{1}{r_{mn1}}}
\]

## 7. Two-stage operational-amplifier file example

The `two_stage_opamp.sp` example is read from `netlists/`. This run uses `eq_circuit=True` and `body="off"`, so MOSFETs are stamped with the equivalent circuit without body-transconductance terms. It also sets `zero_pole=True`; for this larger expression, analytic zero-pole extraction may fail, and that failure is reported in the Markdown output rather than treated as a runtime error.


In [8]:
ssas.solve(
    "netlists/two_stage_opamp.sp",
    "V(vout) / V(in)",
    "two_stage_equivalent_body_off.md",
    source="file",
    eq_circuit=True,
    body="off",
    zero_pole=True,
    show=True,
)

\[
H(s)=\frac{s^{5} E_{1}+s^{4} E_{2}+s^{3} E_{3}+s^{2} E_{4}+s E_{5}+E_{6}}{s^{5} E_{7}+s^{4} E_{8}+s^{3} E_{9}+s^{2} E_{10}+s E_{11}+E_{12}}
\]

\[
C_{mn1}=C_{mn1,gd}+C_{mn1,ds}
\]

\[
C_{mn2}=C_{mn2,gs}+C_{mn2,ds}
\]

\[
C_{mn3}=C_{mn3,gd}+C_{mn3,ds}
\]

\[
C_{mn4}=C_{mn4,gs}+C_{mn4,ds}
\]

...

\[
E_{11}=g_{mn3} \left(C_{mn3,gd} G_{eff2} G_{eff3} G_{eff4}+g_{mp3} C_{mn5,gd} \left(g_{mp2} G_{mn1}-G_{eff2} G_{mn2}\right)\right)+G_{mn4} \left(g_{mp3} C_{eff6} \left(G_{eff1} G_{eff2}-\frac{G_{mn1}}{r_{mn1}}\right)+G_{eff4} \left(g_{mp2} \left(C_{mp2,gd} G_{eff1}+\frac{C_{mn1,ds}}{r_{mn2}}+C_{mn2,ds} G_{mn1}\right)+C_{mp2,gd} -\left(\frac{G_{mn2}}{r_{mn1}}+\frac{G_{mn1}}{r_{mn2}}\right)+C_{mn1,ds} G_{eff3} -\left(\frac{1}{r_{mn1}}+G_{mn1}\right)+C_{eff2} \left(G_{eff1} G_{eff3}-\frac{G_{mn2}}{r_{mn2}}\right)+C_{eff1} \left(G_{eff1} G_{eff2}-\frac{G_{mn1}}{r_{mn1}}\right)+G_{eff2} \left(C_{mn2,ds} -\left(\frac{1}{r_{mn2}}+G_{mn2}\right)+C_{eff3} G_{eff3}\right)\right)\right)-\left(g_{mn5} C_{mn5,gd}+C_{eff4} G_{eff4}+G_{mn4} C_{eff5}\right) \left(\frac{G_{mn1} G_{eff3}}{r_{mn1}}+\frac{G_{eff2} G_{mn2}}{r_{mn2}}\right)+\left(\frac{g_{mp2} G_{mn1}}{r_{mn2}}+G_{eff1} G_{eff2} G_{eff3}\right) \left(g_{mn5} C_{mn5,gd}+C_{eff4} G_{eff4}+G_{mn4} C_{eff5}\right)
\]

\[
E_{12}=G_{mn4} G_{eff4} \left(\frac{g_{mp2} G_{mn1}-G_{eff2} G_{mn2}}{r_{mn2}}+G_{eff3} \left(G_{eff1} G_{eff2}-\frac{G_{mn1}}{r_{mn1}}\right)\right)
\]

\[
\text{zero-pole analysis}
\]

\[
\text{remaining zeros: analytic extraction failed}
\]

\[
\text{remaining poles: analytic extraction failed}
\]